In [0]:
month_folder = "abfss://raw@adeadlsgen2.dfs.core.windows.net/nyc_taxi/yellow/year=2026/month=01"
parquet_name = "yellow_tripdata_2026-01.parquet"

parquet_path = f"{month_folder}/{parquet_name}"
print(parquet_path)

In [0]:
monthly_files = dbutils.fs.ls(month_folder)

for file in monthly_files:
    print(file.name, file.size, "bytes")

In [0]:
trips_df = spark.read.parquet(parquet_path)

trips_df.printSchema()

In [0]:
trip_details = trips_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "trip_distance",
    "fare_amount"
)

display(trip_details.limit(5))

In [0]:
source_row_count = trips_df.count()

print(f"Source file: {parquet_name}")
print(f"Source rows: {source_row_count:,}")

In [0]:
for x in trips_df.schema.fields:
    column_name = x.name
    data_type = x.dataType.simpleString()
    print(f"{column_name} {data_type}")

In [0]:
print(trips_df.schema.fields)
print(f"X = ",{x})

In [0]:
%run ../utilities/schema_definitions

january_trips_df = spark.read.schema(nyc_taxi_explicit_schema).parquet(parquet_path)

In [0]:
source_schema = trips_df.dtypes
explicit_schema = january_trips_df.dtypes

if explicit_schema != source_schema:
    print("Schema from file:", source_schema)
    print("Schema we supplied:", explicit_schema)
    raise ValueError("FAIL: The column names, order, or types differ.")

print(f"PASS: All {len(source_schema)} column names and types match.")

expected_rows = 3_724_889
actual_rows = january_trips_df.count()

if actual_rows != expected_rows:
    raise ValueError(f"FAIL: Expected {expected_rows:,} rows, found {actual_rows:,}.")

print(f"PASS: January row count matches the baseline of {actual_rows:,}.")

In [0]:
month_folder = "abfss://raw@adeadlsgen2.dfs.core.windows.net/nyc_taxi/yellow/year=2026/month=02"
parquet_name = "yellow_tripdata_2026-02.parquet"

february_path = f"{month_folder}/{parquet_name}"
print(february_path)

In [0]:
february_df = spark.read.parquet(february_path)
february_df.printSchema()

In [0]:
# Read the schema stored in February's Parquet file
february_df = spark.read.parquet(february_path)

# Compare each column name and its type with your expected schema
expected_fields = {field.name: field.dataType for field in nyc_taxi_explicit_schema.fields}
actual_fields = {field.name: field.dataType for field in february_df.schema.fields}

if actual_fields == expected_fields:
    print("PASS: All expected column names and types match.")
else:
    for name, expected_type in expected_fields.items():
        if name not in actual_fields:
            print(f"MISSING: {name}")
        elif actual_fields[name] != expected_type:
            actual_type = actual_fields[name]
            print(f"TYPE CHANGED: {name} from {expected_type} to {actual_type}")

    for name in actual_fields:
        if name not in expected_fields:
            print(f"NEW: {name}")

In [0]:
%sql
SELECT
    batch_id,
    pipeline_name,
    source_year,
    source_month,
    start_time,
    end_time,
    timestampdiff(SECOND, start_time, end_time) AS duration_seconds,
    source_row_count,
    target_row_count,
    status,
    error_message
FROM nyc_taxi.control.ingestion_log
ORDER BY start_time DESC;

In [0]:
%sql
SELECT *
FROM nyc_taxi.silver.yellow_taxi_trips_rejected
LIMIT 100;

In [0]:
%sql
SELECT
    rejection_reason,
    COUNT(*) AS rejected_records
FROM nyc_taxi.silver.yellow_taxi_trips_rejected
GROUP BY rejection_reason
ORDER BY rejected_records DESC;

> # Monthly data quality summary


In [0]:
%sql
SELECT
    source_year,
    source_month,
    source_row_count AS bronze_records,
    target_row_count AS accepted_records,
    source_row_count - target_row_count AS rejected_records,
    ROUND(
        target_row_count * 100.0 / NULLIF(source_row_count, 0),
        2
    ) AS acceptance_percentage
FROM nyc_taxi.control.ingestion_log
WHERE pipeline_name = '02_bronze_to_silver'
AND status = 'SUCCESS'
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY source_year, source_month
    ORDER BY start_time DESC, batch_id DESC
) = 1
ORDER BY source_year, source_month;

In [0]:
%sql
SELECT
    rejection_reason,
    COUNT(*) AS rejected_records
FROM nyc_taxi.silver.yellow_taxi_trips_rejected
GROUP BY rejection_reason
ORDER BY rejected_records DESC;